##  OpenTheChests Environment: Overview & Integration

The `OpenTheChests` class implements a **symbolic event-based interactive environment** where agents observe noisy event sequences and must open boxes ("chests") at the right time based on symbolic and temporal cues.

This environment is useful for tasks involving:

* Temporal reasoning
* Event pattern recognition
* Decision-making under uncertainty
* Reinforcement learning (e.g., with Stable Baselines3)

### Class Integration

The `OpenTheChests` environment brings together several key components:

| Component        | Role                                                                  |
| ---------------- | --------------------------------------------------------------------- |
| `Pattern`        | Defines structured event generation instructions for each box         |
| `Parser`         | Translates pattern instructions into concrete events (and adds noise) |
| `Generator`      | Builds and updates event timelines using the `Parser` + `Pattern`     |
| `InteractiveBox` | Represents each box; tracks its state (active, open, ready, etc.)     |
| `OpenTheChests`  | Orchestrates the simulation: advancing time, managing context, etc.   |

Together, they enable the simulation of an interactive world where the **agent must time actions** correctly using noisy symbolic information.

---

### Core Functionality

The environment operates as a reinforcement learning loop with the following high-level methods:

| Method               | Purpose                                                                       |
| -------------------- | ----------------------------------------------------------------------------- |
| `reset()`            | Initializes event timelines and box states for a new episode.                 |
| `step(action)`       | Takes an action, advances time, returns new observations, reward, and done.   |
| `get_observations()` | Returns the current symbolic observation (box states + latest event context). |
| `check_end()`        | Checks whether the episode is done (e.g. all boxes opened, timeout).          |
| `render()`           | (Optional) Visual rendering of the environment state.                         |

---

### How the Environment Works

1. **Initialization**:

   * Each box is associated with a `Pattern`, which defines how and when events should occur.
   * The `Parser` is configured with all valid event/noise types and attributes.
   * The `Generator` builds an event timeline (with noise) for each box.

2. **Event Timeline**:

   * Events are added to each box’s timeline based on its pattern.
   * Events may occur in sequence (e.g., `e2` occurs after `e1`) and are sampled with randomness.

3. **Box Activation**:

   * When an event related to a box occurs, that box may become **active**.
   * A box is **ready to open** only at the correct symbolic + temporal point in the sequence.

4. **Agent Actions**:

   * The agent observes symbolic features of the current event (e.g., `type=A`, `color=red`) and decides **which boxes to open**.
   * If the box is ready, opening it earns a reward (+1). Otherwise, there's a penalty (-1). If neither, no reward (0).

5. **Box Deactivation**

    * If a box is not opened when ready it deactivates.
    * This leads to a randomised timout before its next activation.

6. **Termination Conditions**:

   * All boxes have been opened
   * Too many box deactivations

---

### Observation Format

Observations include:

* `state`: status of each box (`active` and `open` lists)
* `context`: the most recent event, label-encoded

Example in **default mode**:

```python
{
  "state": {
    "active": [True, False, True],
    "open": [False, True, False]
  },
  "context": Event(
      type='1', 
      attr=
          {'bg': 2, 
           'fg': 1}, 
      start=3.973, 
      end=7.973)
}
```

If `stb3=True`, the observation is **flattened into a single dictionary** for compatibility with [Stable Baselines 3](https://stable-baselines3.readthedocs.io/en/master/), which does **not support nested dictionary spaces**. The resulting format merges all information into a single-level dict of numerical values.

---

### Action Format

You can configure the action input format via the `discrete` flag:

| Setting          | Action Format | Description                                                                                         |
| ---------------- | ------------- | --------------------------------------------------------------------------------------------------- |
| `discrete=False` | `List[int]`   | Each entry represents whether a specific box's button is pressed (1 = press, 0 = no press).         |
| `discrete=True`  | `int`         | Action is given as a **binary-encoded integer**, automatically converted to a list inside `step()`. |

Example:

* With `discrete=False` and 3 boxes: `action = [0, 1, 0]` → press only box 1.
* With `discrete=True`: `action = 2` → binary `010` → same as above.

---

### Usage Tips

* Set `verbose=True` for print-based introspection and debugging.
* Use `reset()` to start a fresh environment, followed by repeated calls to `step(action)` in a loop.
* Use `get_observations()` after `step()` or `reset()` to view the current symbolic state.
* Tune `timeout_threshold` to control how long boxes can stay inactive before ending the episode.
* Set `stb3=True` when training with RL frameworks like SB3.


In [1]:
from openthechests.openthechests.src.OpenTheChests import OpenTheChests

# Define available event types and attributes
all_event_types = ["A", "B"]
all_event_attributes = {
    "bg": ["yellow", "blue", "green"],
    "fg": ["red", "black"]
}

# Define noise types and attributes
all_noise_types = ["N"]
all_noise_attributes = {
    "bg": ["grey"],
    "fg": ["grey"]
}

# Define instructions for 2 patterns (boxes)
instructions = [
    [
        {"command": "delay", "parameters": 10},
        {"command": "noise", "parameters": 0.3},
        {"command": "instantiate", "parameters": ("A", {"bg": "yellow", "fg": "red"}, {"mu": 2, "sigma": 1}), "variable_name": "e1"},
        {"command": "instantiate", "parameters": ("A", {"bg": "blue", "fg": "red"}, {"mu": 4, "sigma": 1}), "variable_name": "e2"},
        {"command": "after", "parameters": ["e2", "e1"], "variable_name": "e2", "other": {"gap_dist": {"mu": 2, "sigma": 1}}}
    ],
    [
        {"command": "delay", "parameters": 10},
        {"command": "noise", "parameters": 0.0},
        {"command": "instantiate", "parameters": ("B", {"bg": "green", "fg": "black"}, {"mu": 3, "sigma": 1}), "variable_name": "x1"}
    ]
]

# --- Create environment with default (non-discrete) actions ---
env = OpenTheChests(
    instructions=instructions,
    all_event_types=all_event_types,
    all_event_attributes=all_event_attributes,
    all_noise_types=all_noise_types,
    all_noise_attributes=all_noise_attributes,
    verbose=True,             # Print environment details
    timeout_threshold=10,     # Max deactivations
    stb3=False,               # Use normal observation structure
    discrete=False            # Action is a list of 0/1 per box
)

# --- Reset environment ---
obs = env.reset()

print("----------")
print(" Initial observation (non-SB3 mode):")
print(obs)
print("----------")

# --- Perform an action: try to press button on box 0 only ---
action = [1, 0]  # Press box 0 only
obs, reward, done, info = env.step(action)

print("----------")
print(" Action taken:", action)
print(" Reward received:", reward)
print(" Done:", done)
print(" Updated observation:")
print(obs)
print("----------")

# --- Now create environment with SB3-compatible observations ---
env_sb3 = OpenTheChests(
    instructions=instructions,
    all_event_types=all_event_types,
    all_event_attributes=all_event_attributes,
    all_noise_types=all_noise_types,
    all_noise_attributes=all_noise_attributes,
    verbose=False,
    stb3=True,            # FLATTENED OBSERVATION
    discrete=True         # Action is an INTEGER
)

obs_sb3 = env_sb3.reset()

print("----------")
print(" SB3-compatible observation:")
print(obs_sb3)
print("----------")

# --- Demonstrate discrete action ---
# Action = 1 in binary => '01' -> press only box 1
obs_sb3, reward_sb3, done_sb3, _ = env_sb3.step(1)

print("----------")
print(" Discrete Action = 1 -> binary:", bin(1)[2:].zfill(2))
print(" Reward received:", reward_sb3)
print(" Observation (SB3 format):")
print(obs_sb3)
print("----------")


All event types : ['A', 'B']
All noise types : ['N']
All event attributes : {'bg': ['yellow', 'blue', 'green'], 'fg': ['red', 'black']}
All noise attributes : {'bg': ['grey'], 'fg': ['grey']}
Initialising 2 boxes.
Starting Reset
Sampling new events from pattern 0: [Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=6.604, end=7.877), Event(type='N', attr={'bg': 'grey', 'fg': 'grey'}, start=8.095, end=11.728), Event(type='A', attr={'bg': 'blue', 'fg': 'red'}, start=10.127, end=13.387)]
Sampling new events from pattern 1: [Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=0.979, end=4.689)]
Making one internal step to get context and advance timeline.
Active timeline [Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=6.604, end=7.877), Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=0.979, end=4.689)]
Sampling new events from pattern 1: [Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=9.203, end=11.689)]
The last observed event ends at 4.689


In [2]:
# --- Reset environment ---
obs = env.reset()

# --- Loop until environment ends ---
step_count = 0
print("\nStarting interaction loop...\n")

while not env.done:
    # Press ALL buttons: either as list or integer depending on env config
    if env.discrete:
        # Example: 2 boxes -> pressing all = binary '11' = int(3)
        action = (1 << env.get_num_boxes()) - 1
    else:
        # List of 1s, pressing all buttons
        action = [1] * env.get_num_boxes()

    obs, reward, done, info = env.step(action)
    print(f"Step {step_count}: Action={action}, Reward={reward}, Done={done}")
    step_count += 1

print("\nEnvironment completed.")


Starting Reset
Sampling new events from pattern 0: [Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=5.117, end=7.139), Event(type='N', attr={'bg': 'grey', 'fg': 'grey'}, start=9.21, end=9.293), Event(type='A', attr={'bg': 'blue', 'fg': 'red'}, start=10.139, end=15.139)]
Sampling new events from pattern 1: [Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=5.167, end=7.167)]
Making one internal step to get context and advance timeline.
Active timeline [Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=5.117, end=7.139), Event(type='B', attr={'bg': 'green', 'fg': 'black'}, start=5.167, end=7.167)]
The last observed event ends at 7.139
Advancing _time to 7.139
Observing context Event(type='A', attr={'bg': 'yellow', 'fg': 'red'}, start=5.117, end=7.139)
Activating box 0.
Activating box 1.
Reset Done.

Starting interaction loop...


Start Step
Applying action [1, 1].
Unsuccessful opening of box 0.
Unsuccessful opening of box 1.
Making one internal step to get cont